# Parameter Sweep v1.0
**Purpose:** Identify a defensible calibration of `sigma_r` (HW rate vol) and `rho` (rate-OAS correlation) 
that produces a stress path incidence in the 20–25% range — consistent with historical MBS stress episode 
frequency over a 252-day horizon.

**Grid:** `sigma_r` ∈ {0.008, 0.010, 0.012} × `rho` ∈ {0.35, 0.45, 0.55}  
**Paths per cell:** 1,000 (same as baseline)  
**Fixed parameters:** All other params held at v0.3.3 values (κ=0.15, σ_s=40 bps, T=252)

**Key output metric:** `pct_stress` = fraction of paths where `min_liquidity < 0.2`  
**Target range:** 20–25%

**Usage:**
1. Upload `Oracle_Stale_Gap_v0.3.3.ipynb` to the same directory as this notebook
2. Run all cells
3. Inspect the summary table and heatmap
4. Pick the (sigma_r, rho) cell closest to the target stress rate

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import os, copy, json, importlib, sys, types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

# ── 1. Load simulation core from simulation_v034 notebook ────────────────────
# Extracts the simulation cell and executes it into a module namespace
# so all functions (run_pipeline, generate_stochastic_drivers, etc.) are available
# without touching argparse or the __main__ block.

NB_PATH = "simulation_v034.ipynb"  # must be in the same directory

with open(NB_PATH) as f:
    nb = json.load(f)

sim_src = "".join(nb["cells"][0]["source"])

# Strip argparse / main / __main__ block so exec doesn't fail
cutoff = sim_src.find("def parse_args")
sim_core = sim_src[:cutoff].strip()

# Execute into a dedicated module
sim = types.ModuleType("sim_core")
sys.modules["sim_core"] = sim
exec(compile(sim_core, "sim_core", "exec"), sim.__dict__)
print("Simulation core loaded. Functions available:")
print([k for k in dir(sim) if not k.startswith("_")])

In [ ]:
# ── 2. Sweep configuration ────────────────────────────────────────────────────

SIGMA_R_GRID = [0.008, 0.010, 0.012]   # HW rate vol (daily, absolute)
RHO_GRID     = [0.35,  0.45,  0.55]    # rate-OAS correlation
N_PATHS      = 1000                     # paths per cell
STRESS_THRESH = 0.20                    # min_liquidity threshold for stress classification
TARGET_LOW   = 0.20                     # target stress incidence lower bound
TARGET_HIGH  = 0.25                     # target stress incidence upper bound

print(f"Grid: {len(SIGMA_R_GRID)} × {len(RHO_GRID)} = {len(SIGMA_R_GRID)*len(RHO_GRID)} cells")
print(f"Paths per cell: {N_PATHS}")
print(f"Total paths: {N_PATHS * len(SIGMA_R_GRID) * len(RHO_GRID):,}")
print(f"Target stress incidence: {TARGET_LOW*100:.0f}–{TARGET_HIGH*100:.0f}%")

In [ ]:
# ── 3. Per-cell runner ────────────────────────────────────────────────────────

def run_cell(sigma_r: float, rho: float, n_paths: int, base_params: dict) -> dict:
    """
    Run n_paths stochastic paths for a given (sigma_r, rho) combination.
    Returns a dict of summary statistics.
    """
    p = copy.deepcopy(base_params)
    p["stochastic"]["hw"]["sigma"] = sigma_r
    p["stochastic"]["corr"]["rho_rate_oas"] = rho

    T   = p["T_days"]
    ocs = p["oracle_staleness_stochastic"]
    rng = np.random.default_rng(p["seed"])

    min_liqs      = []
    max_drawdowns = []
    emergency_days_list = []
    max_oas_list  = []
    max_liqs_list = []

    for k in range(n_paths):
        seed_k = int(p["seed"] + 10_000 + k)
        drivers = sim.generate_stochastic_drivers(T, p, seed=seed_k)

        oracle_cfg = None
        if ocs.get("enabled", True) and (rng.uniform() < float(ocs["path_probability"])):
            oracle_cfg = {
                "enabled": True,
                "stale_start_day": ocs["stale_start_day"],
                "stale_days": ocs["stale_days"],
                "freeze_fields": ocs["freeze_fields"],
            }

        garch_seed = seed_k + 12345
        risk, daily, _ = sim.run_pipeline(drivers, p,
                                           oracle_stale=oracle_cfg,
                                           garch_seed=garch_seed)

        min_l = float(np.nanmin(risk["liq"].values))
        max_o = float(np.nanmax(risk["oas"].values))
        max_liq_daily = float(np.nanmax(daily["liquidations"].values)) if len(daily) else 0.0

        nav   = risk["NAV"].values
        peak  = np.maximum.accumulate(nav)
        dd    = float(np.nanmin((nav / np.maximum(peak, 1e-12)) - 1.0))
        em    = int(np.sum(risk["State"].values == "EMERGENCY_UNWIND"))

        min_liqs.append(min_l)
        max_drawdowns.append(dd)
        emergency_days_list.append(em)
        max_oas_list.append(max_o)
        max_liqs_list.append(max_liq_daily)

    min_liqs      = np.array(min_liqs)
    max_drawdowns = np.array(max_drawdowns)
    emergency_days_arr = np.array(emergency_days_list)
    max_oas_arr   = np.array(max_oas_list)

    pct_stress  = (min_liqs < STRESS_THRESH).mean()
    med_dd      = float(np.median(max_drawdowns))
    p95_dd      = float(np.percentile(max_drawdowns, 95))
    p99_dd      = float(np.percentile(max_drawdowns, 99))
    pct_emerg   = (emergency_days_arr > 0).mean()
    max_emerg   = int(emergency_days_arr.max())
    med_oas     = float(np.median(max_oas_arr[min_liqs >= STRESS_THRESH])) if (min_liqs >= STRESS_THRESH).any() else float('nan')

    in_target   = TARGET_LOW <= pct_stress <= TARGET_HIGH

    return {
        "sigma_r":    sigma_r,
        "rho":        rho,
        "pct_stress": pct_stress,
        "med_drawdown_pct": med_dd * 100,
        "p95_drawdown_pct": p95_dd * 100,
        "p99_drawdown_pct": p99_dd * 100,
        "pct_emergency":    pct_emerg,
        "max_emergency_days": max_emerg,
        "stable_med_oas_bps": med_oas,
        "in_target":  in_target,
    }

print("Runner defined.")

In [ ]:
# ── 4. Execute the sweep ──────────────────────────────────────────────────────
# Runtime: ~2–4 min on a standard laptop (9,000 paths total)

base_params = copy.deepcopy(sim.PARAMS)
results = []

total = len(SIGMA_R_GRID) * len(RHO_GRID)
done  = 0

for sigma_r in SIGMA_R_GRID:
    for rho in RHO_GRID:
        done += 1
        print(f"[{done}/{total}] sigma_r={sigma_r:.3f}  rho={rho:.2f} ...", end=" ", flush=True)
        row = run_cell(sigma_r, rho, N_PATHS, base_params)
        results.append(row)
        flag = "✅ IN TARGET" if row["in_target"] else ""
        print(f"stress={row['pct_stress']*100:.1f}%  med_dd={row['med_drawdown_pct']:.1f}%  emerg={row['pct_emergency']*100:.1f}%  {flag}")

df_results = pd.DataFrame(results)
print("\nSweep complete.")

In [ ]:
# ── 5. Summary table ──────────────────────────────────────────────────────────

display_cols = {
    "sigma_r":            "σ_r",
    "rho":                "ρ",
    "pct_stress":         "Stress %",
    "med_drawdown_pct":   "Med DD %",
    "p95_drawdown_pct":   "p95 DD %",
    "p99_drawdown_pct":   "p99 DD %",
    "pct_emergency":      "Emerg %",
    "max_emergency_days": "Max Emerg Days",
    "in_target":          "In Target?",
}

tbl = df_results[list(display_cols.keys())].copy()
tbl.columns = list(display_cols.values())
tbl["Stress %"]  = tbl["Stress %"].apply(lambda x: f"{x*100:.1f}%")
tbl["Med DD %"]  = tbl["Med DD %"].apply(lambda x: f"{x:.2f}%")
tbl["p95 DD %"]  = tbl["p95 DD %"].apply(lambda x: f"{x:.2f}%")
tbl["p99 DD %"]  = tbl["p99 DD %"].apply(lambda x: f"{x:.2f}%")
tbl["Emerg %"]   = tbl["Emerg %"].apply(lambda x: f"{x*100:.1f}%")
tbl["In Target?"] = tbl["In Target?"].apply(lambda x: "✅" if x else "")

print(f"\nParameter Sweep Results (target stress: {TARGET_LOW*100:.0f}–{TARGET_HIGH*100:.0f}%)")
print("=" * 80)
display(tbl.to_string(index=False))

In [ ]:
# ── 6. Heatmap: stress incidence across grid ──────────────────────────────────

C_BG   = "#F7F9FC"
C_DARK = "#1A1A2E"

stress_grid = np.array(df_results["pct_stress"]).reshape(len(SIGMA_R_GRID), len(RHO_GRID))

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

# Colour: green in target band, red outside
cmap = plt.cm.RdYlGn_r
norm = mcolors.TwoSlopeNorm(vmin=0.0, vcenter=0.225, vmax=0.60)
im = ax.imshow(stress_grid, cmap=cmap, norm=norm, aspect="auto")

ax.set_xticks(range(len(RHO_GRID)))
ax.set_xticklabels([f"ρ = {r:.2f}" for r in RHO_GRID], fontsize=10)
ax.set_yticks(range(len(SIGMA_R_GRID)))
ax.set_yticklabels([f"σᵣ = {s:.3f}" for s in SIGMA_R_GRID], fontsize=10)
ax.set_xlabel("Rate–OAS Correlation (ρ)", fontsize=11)
ax.set_ylabel("HW Rate Volatility (σᵣ)", fontsize=11)

# Annotate each cell
for i in range(len(SIGMA_R_GRID)):
    for j in range(len(RHO_GRID)):
        val = stress_grid[i, j]
        in_tgt = TARGET_LOW <= val <= TARGET_HIGH
        label = f"{val*100:.1f}%"
        if in_tgt:
            label += "\n✅"
        ax.text(j, i, label, ha="center", va="center",
                fontsize=11, fontweight="bold",
                color="white" if val > 0.40 or val < 0.12 else C_DARK)

# Target band indicator
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Stress Path Incidence", fontsize=9)
cb.ax.axhline(TARGET_LOW,  color="green", lw=1.5, ls="--")
cb.ax.axhline(TARGET_HIGH, color="green", lw=1.5, ls="--")

ax.set_title("Stress Path Incidence Across Parameter Grid\n"
             f"(Target band: {TARGET_LOW*100:.0f}–{TARGET_HIGH*100:.0f}%  |  κ=0.15, σ_s=40 bps, T=252)",
             fontsize=12, fontweight="bold", color=C_DARK, pad=12)

plt.tight_layout()
plt.savefig("sweep_heatmap_stress.png", dpi=180, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Heatmap saved: sweep_heatmap_stress.png")

In [ ]:
# ── 7. Drawdown heatmap ───────────────────────────────────────────────────────

dd_grid = np.array(df_results["med_drawdown_pct"]).reshape(len(SIGMA_R_GRID), len(RHO_GRID))

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

im2 = ax.imshow(dd_grid, cmap=plt.cm.YlOrRd, aspect="auto")
ax.set_xticks(range(len(RHO_GRID)))
ax.set_xticklabels([f"ρ = {r:.2f}" for r in RHO_GRID], fontsize=10)
ax.set_yticks(range(len(SIGMA_R_GRID)))
ax.set_yticklabels([f"σᵣ = {s:.3f}" for s in SIGMA_R_GRID], fontsize=10)
ax.set_xlabel("Rate–OAS Correlation (ρ)", fontsize=11)
ax.set_ylabel("HW Rate Volatility (σᵣ)", fontsize=11)

for i in range(len(SIGMA_R_GRID)):
    for j in range(len(RHO_GRID)):
        ax.text(j, i, f"{dd_grid[i,j]:.2f}%",
                ha="center", va="center", fontsize=11,
                fontweight="bold", color=C_DARK)

cb2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
cb2.set_label("Median Peak NAV Drawdown (%)", fontsize=9)

ax.set_title("Median Peak NAV Drawdown Across Parameter Grid",
             fontsize=12, fontweight="bold", color=C_DARK, pad=12)

plt.tight_layout()
plt.savefig("sweep_heatmap_drawdown.png", dpi=180, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Heatmap saved: sweep_heatmap_drawdown.png")

In [ ]:
# ── 8. Recommendation ─────────────────────────────────────────────────────────

in_target = df_results[df_results["in_target"]]

print("=" * 60)
print("CELLS IN TARGET RANGE (stress 20–25%)")
print("=" * 60)

if len(in_target) == 0:
    # Fall back to closest cell
    df_results["dist_to_target"] = df_results["pct_stress"].apply(
        lambda x: 0 if TARGET_LOW <= x <= TARGET_HIGH else min(abs(x - TARGET_LOW), abs(x - TARGET_HIGH))
    )
    closest = df_results.sort_values("dist_to_target").iloc[0]
    print(f"No cell in target range. Closest: sigma_r={closest.sigma_r:.3f}, rho={closest.rho:.2f}")
    print(f"  Stress: {closest.pct_stress*100:.1f}%")
    print(f"  Med drawdown: {closest.med_drawdown_pct:.2f}%")
else:
    for _, row in in_target.iterrows():
        print(f"\nsigma_r = {row.sigma_r:.3f}  |  rho = {row.rho:.2f}")
        print(f"  Stress incidence:    {row.pct_stress*100:.1f}%")
        print(f"  Median drawdown:     {row.med_drawdown_pct:.2f}%")
        print(f"  p95 drawdown:        {row.p95_drawdown_pct:.2f}%")
        print(f"  Emergency paths:     {row.pct_emergency*100:.1f}%")
        print(f"  Max emergency days:  {row.max_emergency_days}")

print("\n" + "=" * 60)
print("Next step: update v0.3.3 PARAMS with chosen sigma_r and rho,")
print("rerun full stochastic pass, regenerate MC figures, update paper.")
print("=" * 60)

# Save results to CSV for reference
df_results.to_csv("sweep_results.csv", index=False)
print("\nFull results saved: sweep_results.csv")